# Exercise 9.4 — For whom is the model worst? (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day9/day6-04-subgroups.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

> **A suggested addition, not part of the original module**
>
> This exercise has no page in the original version of the module. Exercises
> 9.1 to 9.3 explain the model as a whole and one patient at a time; this one
> asks whether it works equally well for different groups of patients. Accept,
> rewrite or remove it; nothing else depends on it.

One test AUC describes the model for all the test patients together. The
model can still serve some groups much worse than others, and the overall
number will not show it. Here we split the test patients by age and by BMI and
compare.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### The model

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

The same data ([`data/diabetes_clean.csv`](data/diabetes_clean.csv)) and
random forest as Exercises 9.1 to 9.3. As always, 80% of the people are used
for training and 20% are held back as the test set.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["pregnant", "glucose", "mass", "pedigree", "age", "diabetes"]]
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

predictors = ["pregnant", "glucose", "mass", "pedigree", "age"]

train, test = train_test_split(diabetes, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=500, random_state=1)
rf = rf.fit(train[predictors], train["diabetes"])

test_preds = test.copy()
test_preds["prob"] = rf.predict_proba(test[predictors])[:, 1]   # probability of diabetes
test_preds["pred"] = rf.predict(test[predictors])               # predicted class, threshold 0.5

Evaluate it once on all the test rows (threshold 0.5):

In [ ]:
print("accuracy   ", round(accuracy_score(test_preds["diabetes"], test_preds["pred"]), 3))
print("sensitivity", round(recall_score(test_preds["diabetes"], test_preds["pred"]), 3))
print("AUC        ", round(roc_auc_score(test_preds["diabetes"], test_preds["prob"]), 3))

Accuracy 0.713, sensitivity 0.474, AUC 0.794. These are the overall numbers
we will now break down.

------------------------------------------------------------------------

### Part 1 — Performance by age group

Put the test patients into three age groups. The loop takes one group at a
time and records the number of patients, the patients with diabetes, how many
of those the model catches, the accuracy, the sensitivity and the AUC.

In [ ]:
test_preds["age_group"] = pd.cut(test_preds["age"], bins=[20, 29, 39, np.inf],
                                 labels=["21-29", "30-39", "40 and over"])

rows = []
for group, patients in test_preds.groupby("age_group", observed=True):
    rows.append({"age_group": group,
                 "patients": len(patients),
                 "with_diabetes": patients["diabetes"].sum(),
                 "caught": ((patients["diabetes"] == 1) & (patients["pred"] == 1)).sum(),
                 "accuracy": accuracy_score(patients["diabetes"], patients["pred"]),
                 "sensitivity": recall_score(patients["diabetes"], patients["pred"]),
                 "roc_auc": roc_auc_score(patients["diabetes"], patients["prob"])})

pd.DataFrame(rows).round(3)

**Question!** Which age group does the model serve worst?

It depends on the measure. In the youngest group the accuracy looks fine
(0.775), but the model catches only 10 of the 26 people with diabetes
(sensitivity 0.385); most young people do not have diabetes, so being right
about them keeps the accuracy up. The oldest group has the lowest accuracy
(0.556) and the lowest AUC (0.669), and the model catches only 9 of its 20
cases. None of this shows in the overall numbers.

The plot shows the predicted probabilities in each age group, split by whether
the patient really has diabetes. Most of the young people with diabetes sit
below the 0.5 line.

In [ ]:
groups = ["21-29", "30-39", "40 and over"]
fig, ax = plt.subplots(figsize=(7, 3.5))
for shift, outcome, colour in [(-0.2, 0, "tab:blue"), (0.2, 1, "tab:red")]:
    for i, group in enumerate(groups):
        values = test_preds.loc[(test_preds["age_group"] == group) &
                                (test_preds["diabetes"] == outcome), "prob"]
        ax.boxplot(values, positions=[i + shift], widths=0.3, showfliers=False)
        ax.scatter(np.full(len(values), i + shift), values, color=colour, alpha=0.5,
                   label=("diabetes" if outcome == 1 else "no diabetes") if i == 0 else None)
ax.axhline(0.5, color="black", linestyle="--")
ax.set_xticks(range(len(groups)))
ax.set_xticklabels(groups)
ax.set_xlabel("Age group")
ax.set_ylabel("Predicted probability of diabetes")
ax.legend(title="Diabetes", loc="center left", bbox_to_anchor=(1, 0.5))
fig.tight_layout()
plt.show()

------------------------------------------------------------------------

### Part 2 — By BMI, and how far to trust a small group

**Your turn.** Do the same with two BMI groups: under 30, and 30 or over (the
usual cut-off for obesity).

> **Stuck?**
>
> BMI is the column `mass`, and the cut-off is 30. Group by the new column,
> `"bmi_group"`.

> **Worked answer**

In [ ]:
test_preds["bmi_group"] = np.where(test_preds["mass"] < 30, "under 30", "30 and over")

rows = []
for group, patients in test_preds.groupby("bmi_group"):
    rows.append({"bmi_group": group,
                 "patients": len(patients),
                 "with_diabetes": patients["diabetes"].sum(),
                 "caught": ((patients["diabetes"] == 1) & (patients["pred"] == 1)).sum(),
                 "accuracy": accuracy_score(patients["diabetes"], patients["pred"]),
                 "sensitivity": recall_score(patients["diabetes"], patients["pred"]),
                 "roc_auc": roc_auc_score(patients["diabetes"], patients["prob"])})

pd.DataFrame(rows).round(3)

> BMI 30 and over: 99 patients, 45 with diabetes, 26 caught (sensitivity
> 0.578), accuracy 0.707, AUC 0.796. Under 30: 51 patients, 12 with diabetes,
> 1 caught (sensitivity 0.083), accuracy 0.725, AUC 0.732. The same pattern as
> the youngest age group: in the lower-risk group the model misses most of the
> cases.

**How far can we trust these numbers?** The sensitivity of 0.083 in the
BMI-under-30 group rests on 12 people: 1 caught out of 12. One more caught
would double it to 0.167. With so few people, one patient moves the number a
long way, and a different random split could tell a different story.
Differences between small groups are a reason to look again with more data,
not a finding. That is why the counts always go beside the metrics.

> **Key points**
>
> - The overall numbers can hide large differences between groups of patients.
> - Which group looks worst depends on the measure: accuracy, sensitivity and AUC answer different questions.
> - Always report how many patients, and how many with the outcome, each group's numbers rest on. A few people give unstable numbers.